In [2]:
!pip install ollama

### 모델 성능 체크 및 프롬프터 검증

In [15]:
import json
import ollama

# 1. 테스트할 사용자 입력 문장
#user_input = "운동 후 단백질 보충할 음식을 10,000원 이내에서 추천해줘"
#user_input = "야식을 먹고 싶은데 내일 아침 얼굴 안 붓는 음식으로 6,000원 이내에서 추천해줘"
#user_input = "5천 원으로 배부르게 먹을 수 있는 음식 추천해 줘."
user_input = "맛있는 거 아무거나 추천해줘"

# 2. LLM에 전달할 프롬프트 구성
system_prompt = """
너는 편의점 상품 추천 시스템의 '사용자 요청 해석기'야. 
사용자의 입력에서 조건(카테고리, 예산, 영양 목표)을 추출해서 오직 지정된 JSON 형식으로만 응답해.

[카테고리 종류]
식사류, 사이드류, 디저트류, 음료류

[영양 목표 종류]
- "protein": 단백질, 운동, 근육 관련
- "low_sodium": 나트륨, 부기, 야식, 덜 짜게 관련
- "none": 별도 영양 조건 없음

[need_more_info 판단 기준]
- 예산(max_price)이나 구체적인 조건이 전혀 없어서 추천 방향을 잡기 어려울 때만 true
- 예산이나 영양 목표 중 하나라도 명확히 제시되어 있으면 false

[입력 및 출력 예시 (Few-shot)]
입력: "맛있는 거 아무거나 추천해줘"
응답:
{
  "categories": [],
  "max_price": null,
  "nutrition_focus": "none",
  "need_more_info": true
}

입력: "운동 후 먹을 1만원 이하 단백질 조합"
응답:
{
  "categories": ["식사류", "음료류"],
  "max_price": 10000,
  "nutrition_focus": "protein",
  "need_more_info": false
}

[출력 규칙]
- 입력에 금액/예산이 언급되지 않았으면 max_price는 절대로 임의의 숫자를 넣지 말고 반드시 null로 출력해.
- 오직 valid한 JSON만 출력해.
"""

# 3. Llama 3.2 3B 모델로 요청 보내기 (format='json' 옵션으로 JSON 출력 강제)
response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ],
    format="json",  # Ollama가 반환 형식을 JSON으로 고정해주는 유용한 옵션입니다!
)

# 4. 결과 출력
print("=== LLM 응답 원문 ===")
print(response["message"]["content"])

# 5. 파이썬 JSON 객체로 잘 변환되는지 검증
try:
    parsed_json = json.loads(response["message"]["content"])
    print("\n=== JSON 파싱 성공! ===")
    print(f"추출된 예산: {parsed_json.get('max_price')}원")
    print(f"영양 목표: {parsed_json.get('nutrition_focus')}")
except Exception as e:
    print(f"\nJSON 파싱 실패: {e}")

=== LLM 응답 원문 ===
{
  "categories": [],
  "max_price": null,
  "nutrition_focus": "none",
  "need_more_info": true
}

=== JSON 파싱 성공! ===
추출된 예산: None원
영양 목표: none


### 위 프롬프터에 시연용 임시 데이터 연결

In [19]:
import json
import ollama

# ==========================================
# 1. 테스트할 사용자 입력 문장 (여기서 문장을 바꿔가며 테스트하세요!)
# ==========================================
user_input = "운동 후 단백질 보충할 음식을 10,000원 이내에서 추천해줘"
# user_input = "야식을 먹고 싶은데 내일 아침 얼굴 안 붓는 음식으로 6,000원 이내에서 추천해줘"
# user_input = "맛있는 거 아무거나 추천해줘"


# ==========================================
# 2. LLM 조건 추출 (프롬프트 실행)
# ==========================================
system_prompt = """
너는 편의점 상품 추천 시스템의 '사용자 요청 해석기'야. 
사용자의 입력에서 조건(카테고리, 예산, 영양 목표)을 추출해서 오직 지정된 JSON 형식으로만 응답해.

[영양 목표 종류]
- "protein": 단백질, 운동, 근육 관련
- "low_sodium": 나트륨, 부기, 야식, 덜 짜게 관련
- "none": 별도 영양 조건 없음

[need_more_info 판단 기준]
- 예산(max_price)이나 구체적인 조건이 전혀 없어서 추천 방향을 잡기 어려울 때만 true
- 예산이나 영양 목표 중 하나라도 명확히 제시되어 있으면 false

[입력 및 출력 예시 (Few-shot)]
입력: "맛있는 거 아무거나 추천해줘"
응답:
{
  "categories": [],
  "max_price": null,
  "nutrition_focus": "none",
  "need_more_info": true
}

입력: "운동 후 먹을 1만원 이하 단백질 조합"
응답:
{
  "categories": ["식사류", "음료류"],
  "max_price": 10000,
  "nutrition_focus": "protein",
  "need_more_info": false
}

[출력 규칙]
- 입력에 금액/예산이 언급되지 않았으면 max_price는 절대로 임의의 숫자를 넣지 말고 반드시 null로 출력해.
- 오직 valid한 JSON만 출력해.
"""

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ],
    format="json",
)

parsed_json = json.loads(response["message"]["content"])


# ==========================================
# 3. 파이썬 계산 및 추천 엔진 (Dummy DB 연동)
# ==========================================
dummy_db = [
    {"id": 1, "name": "혜자로운 제육도시락", "category": "식사류", "price": 4500, "protein": 22, "sodium": 890},
    {"id": 2, "name": "훈제란 2입", "category": "사이드류", "price": 2200, "protein": 12, "sodium": 210},
    {"id": 3, "name": "저당 고단백 두유", "category": "음료류", "price": 1800, "protein": 10, "sodium": 110},
    {"id": 4, "name": "매콤 떡볶이", "category": "식사류", "price": 3800, "protein": 6, "sodium": 1250},
    {"id": 5, "name": "초코 롤케이크", "category": "디저트류", "price": 2500, "protein": 3, "sodium": 180},
]

max_price = parsed_json.get("max_price")
nutrition_focus = parsed_json.get("nutrition_focus")
need_more_info = parsed_json.get("need_more_info")

print(f"📥 사용자 입력: \"{user_input}\"")

if need_more_info:
    print("\n💬 [시스템 안내] 예산이나 원하시는 조건(단백질, 야식 등)을 조금 더 자세히 말씀해주세요!")
else:
    print(f"🔍 [추출된 조건] 최대 예산: {max_price}원 | 강조 영양: {nutrition_focus}\n")
    
    recommended_combinations = []
    
    for i in range(len(dummy_db)):
        for j in range(i + 1, len(dummy_db)):
            item1 = dummy_db[i]
            item2 = dummy_db[j]
            
            total_price = item1["price"] + item2["price"]
            total_protein = item1["protein"] + item2["protein"]
            total_sodium = item1["sodium"] + item2["sodium"]
            
            if max_price is None or total_price <= max_price:
                recommended_combinations.append({
                    "items": [item1["name"], item2["name"]],
                    "total_price": total_price,
                    "total_protein": total_protein,
                    "total_sodium": total_sodium
                })
    
    if nutrition_focus == "protein":
        recommended_combinations.sort(key=lambda x: x["total_protein"], reverse=True)
    elif nutrition_focus == "low_sodium":
        recommended_combinations.sort(key=lambda x: x["total_sodium"])
        
    print("=== 🎯 추천 조합 TOP 2 ===")
    for idx, combo in enumerate(recommended_combinations[:2], 1):
        print(f"[{idx}위 조합] {' + '.join(combo['items'])}")
        print(f" - 총 가격: {combo['total_price']}원")
        print(f" - 총 단백질: {combo['total_protein']}g | 총 나트륨: {combo['total_sodium']}mg\n")

📥 사용자 입력: "운동 후 단백질 보충할 음식을 10,000원 이내에서 추천해줘"
🔍 [추출된 조건] 최대 예산: 10000원 | 강조 영양: protein

=== 🎯 추천 조합 TOP 2 ===
[1위 조합] 혜자로운 제육도시락 + 훈제란 2입
 - 총 가격: 6700원
 - 총 단백질: 34g | 총 나트륨: 1100mg

[2위 조합] 혜자로운 제육도시락 + 저당 고단백 두유
 - 총 가격: 6300원
 - 총 단백질: 32g | 총 나트륨: 1000mg



### 위 로직에서 추천이유 생성 로직 추가

In [21]:
import json
import ollama

# ==========================================
# 1. 사용자 입력 (테스트 문장)
# ==========================================
user_input = "운동 후 단백질 보충할 음식을 10,000원 이내에서 추천해줘"


# ==========================================
# 2. LLM 조건 추출 (1차 LLM 호출)
# ==========================================
system_prompt_extract = """
너는 편의점 상품 추천 시스템의 '사용자 요청 해석기'야. 
사용자의 입력에서 조건(카테고리, 예산, 영양 목표)을 추출해서 오직 지정된 JSON 형식으로만 응답해.

[영양 목표 종류]
- "protein": 단백질, 운동, 근육 관련
- "low_sodium": 나트륨, 부기, 야식, 덜 짜게 관련
- "none": 별도 영양 조건 없음

[need_more_info 판단 기준]
- 예산(max_price)이나 구체적인 조건이 전혀 없어서 추천 방향을 잡기 어려울 때만 true
- 예산이나 영양 목표 중 하나라도 명확히 제시되어 있으면 false

[입력 및 출력 예시 (Few-shot)]
입력: "맛있는 거 아무거나 추천해줘"
응답:
{
  "categories": [],
  "max_price": null,
  "nutrition_focus": "none",
  "need_more_info": true
}

[출력 규칙]
- 입력에 금액/예산이 언급되지 않았으면 max_price는 절대로 임의의 숫자를 넣지 말고 반드시 null로 출력해.
- 오직 valid한 JSON만 출력해.
"""

response_extract = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": system_prompt_extract},
        {"role": "user", "content": user_input},
    ],
    format="json",
)

parsed_json = json.loads(response_extract["message"]["content"])


# ==========================================
# 3. 파이썬 정확 수치 계산 (Dummy DB)
# ==========================================
dummy_db = [
    {"id": 1, "name": "혜자로운 제육도시락", "category": "식사류", "price": 4500, "protein": 22, "sodium": 890},
    {"id": 2, "name": "훈제란 2입", "category": "사이드류", "price": 2200, "protein": 12, "sodium": 210},
    {"id": 3, "name": "저당 고단백 두유", "category": "음료류", "price": 1800, "protein": 10, "sodium": 110},
    {"id": 4, "name": "매콤 떡볶이", "category": "식사류", "price": 3800, "protein": 6, "sodium": 1250},
    {"id": 5, "name": "초코 롤케이크", "category": "디저트류", "price": 2500, "protein": 3, "sodium": 180},
]

max_price = parsed_json.get("max_price")
nutrition_focus = parsed_json.get("nutrition_focus")
need_more_info = parsed_json.get("need_more_info")

print(f"📥 사용자 입력: \"{user_input}\"\n")

if need_more_info:
    print("💬 [시스템 안내] 예산이나 원하시는 조건(단백질, 야식 등)을 조금 더 자세히 말씀해주세요!")
else:
    recommended_combinations = []
    
    for i in range(len(dummy_db)):
        for j in range(i + 1, len(dummy_db)):
            item1, item2 = dummy_db[i], dummy_db[j]
            total_price = item1["price"] + item2["price"]
            total_protein = item1["protein"] + item2["protein"]
            total_sodium = item1["sodium"] + item2["sodium"]
            
            if max_price is None or total_price <= max_price:
                recommended_combinations.append({
                    "items": [item1["name"], item2["name"]],
                    "total_price": total_price,
                    "total_protein": total_protein,
                    "total_sodium": total_sodium
                })
    
    if nutrition_focus == "protein":
        recommended_combinations.sort(key=lambda x: x["total_protein"], reverse=True)
    elif nutrition_focus == "low_sodium":
        recommended_combinations.sort(key=lambda x: x["total_sodium"])
        
    top1_combo = recommended_combinations[0]
    
    
    # ==========================================
    # 4. 추천 이유 생성 (2차 LLM 호출) - [신규 추가!]
    # ==========================================
    top1_info_text = f"""
    - 추천 조합: {' + '.join(top1_combo['items'])}
    - 총 가격: {top1_combo['total_price']}원
    - 총 단백질: {top1_combo['total_protein']}g
    - 총 나트륨: {top1_combo['total_sodium']}mg
    """
    
    system_prompt_reason = """
    너는 편의점 식품 큐레이터야. 
    제시된 [사용자 질문]과 코드가 계산한 [추천 조합 정보]를 바탕으로, 해당 조합을 왜 추천하는지 친절하고 매력적인 1~2문장의 추천 사유를 작성해줘.
    
    [주의 사항]
    - 영어 단어나 한자를 섞지 말고, 반드시 자연스럽고 쉬운 한국어로만 작성해.
    - 새로운 숫자나 가격, 영양성분을 절대로 지어내지 말고 제시된 정보만 언급해.
    - 새로운 숫자나 가격, 영양성분을 절대로 지어내지 말고 제시된 정보만 언급해.
    - 친근한 톤앤매너로 작성해.
    """
    
    user_prompt_reason = f"""
    [사용자 질문]: {user_input}
    [추천 조합 정보]:
    {top1_info_text}
    """
    
    response_reason = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": system_prompt_reason},
            {"role": "user", "content": user_prompt_reason},
        ],
    )
    
    # ==========================================
    # 5. 최종 결과 출력
    # ==========================================
    print("=== 🎯 1위 추천 카드 ===")
    print(f"📦 조합: {' + '.join(top1_combo['items'])}")
    print(f"💰 총 가격: {top1_combo['total_price']}원")
    print(f"📊 영양성분: 단백질 {top1_combo['total_protein']}g | 나트륨 {top1_combo['total_sodium']}mg")
    print(f"\n💡 [AI 추천 이유]\n{response_reason['message']['content']}")

📥 사용자 입력: "운동 후 단백질 보충할 음식을 10,000원 이내에서 추천해줘"

=== 🎯 1위 추천 카드 ===
📦 조합: 혜자로운 제육도시락 + 훈제란 2입
💰 총 가격: 6700원
📊 영양성분: 단백질 34g | 나트륨 1100mg

💡 [AI 추천 이유]
운동 후 단백질 보충에 도움이 될 음식 추천!

제육도시락과 훈제란 2입을 함께 선택해서 추천해드릴게요. 제육도시락은 고마스인 고등fish의 고마스 가루를 sử여 만든 제육이기 때문에 단백질 보충에 좋고, 훈제란은 비타민 B가 풍부해 신체를 에너지로 전환에 도움이 됩니다. 따라서 제육도시락과 훈제란을 함께 먹으면 단백질 보충과 비타민 공급을同時 할 수 있습니다.


### 식약처 데이터 정제 (공유 엑셀시트에 분류한 그대로 적용)
#### => 그래서 매핑이 적게 됨.

In [26]:
import pandas as pd

# 1. 식약처 파일 불러오기
file_path = "20260929_가공식품DB_323197건 (1).xlsx"  # 실제 파일 경로 및 이름으로 수정 (.csv인 경우 pd.read_csv 사용)
df = pd.read_excel(file_path)

# 2. 새 이미지 표 기준: 소분류 -> 4대 카테고리 매핑 딕셔너리 생성
category_map = {}

# [식사류]
meal_items = [
    "즉석조리식품", "즉석섭취식품", "건면", "숙면", "생면", "유탕면", "만두", "떡류",
    "곡류가공품", "신선편의식품", "카레(커리)", "생식제품", "생식함유제품", "묵류",
    "기타 농산가공품", "열량 및 영양공급용 식품"
]
for item in meal_items:
    category_map[item] = "식사류"

# [사이드류]
side_items = [
    "양념육", "분쇄가공육제품", "갈비가공품", "햄", "프레스햄", "생햄", "발효소시지", "소시지",
    "어육소시지", "조미건어포", "건어포", "기타 건포류", "치즈", "가공치즈", "모조치즈",
    "두부", "가공두부", "두류가공품", "청국장", "땅콩버터", "기타 동물성가공식품", "기타 어육가공품",
    "김치", "절임식품", "소스", "마요네즈", "토마토케첩"
]
for item in side_items:
    category_map[item] = "사이드류"

# [디저트류]
dessert_items = [
    "빵류", "과자", "캔디류", "초콜릿가공품", "초콜릿", "준초콜릿", "밀크초콜릿", "기타 코코아가공품",
    "아이스크림", "비유지방아이스크림", "저지방아이스크림", "땅콩 또는 견과류가공품"
]
for item in dessert_items:
    category_map[item] = "디저트류"

# [음료류]
drink_items = [
    "혼합음료", "과·채주스", "과·채음료", "액상차", "침출차", "고형차", "음료베이스", "탄산수",
    "기타발효음료", "농축과·채즙(또는 과·채분)", "가공두유", "원액두유", "우유", "강화유유", "가공유",
    "유당분해유", "유산균첨가유", "발효유", "농후발효유", "농후크림발효유", "크림발효유", "탈지농축우유",
    "환원유", "가공유크림", "유크림", "가공연유", "가당연유", "수분 및 전해질보충용 조제식품"
]
for item in drink_items:
    category_map[item] = "음료류"


# 3. 식약처 파일의 실제 소분류 컬럼명 입력 (예: '식품소분류명')
subcat_col = "식품소분류명"

# 이미지 표에 표기된 소분류만 선택 (표에 없는 불필요 항목 자동 제거)
filtered_df = df[df[subcat_col].isin(category_map.keys())].copy()

# 4. 4대 카테고리('category') 열 새로 생성
filtered_df["category"] = filtered_df[subcat_col].map(category_map)

# 5. 추출할 핵심 6대 영양성분 및 주요 컬럼 지정
target_columns = [
    "category",      # 새로 추가된 4대 대분류 (식사류, 사이드류, 디저트류, 음료류)
    subcat_col,      # 식약처 소분류명
    "식품명",        # 상품명
    "제조사명",      # 제조사명
    "식품중량",      # 식품중량
    "에너지(kcal)",    # 열량
    "단백질(g)",     # 단백질
    "나트륨(mg)",    # 나트륨
    "탄수화물(g)",   # 탄수화물
    "당류(g)",       # 당류
    "지방(g)"        # 지방
]

# 엑셀 파일에 실제로 존재하는 컬럼만 남기기
existing_cols = [c for c in target_columns if c in filtered_df.columns]
final_df = filtered_df[existing_cols]

# 6. 정제 결과 CSV 파일로 저장
final_df.to_csv("식약처_영양DB_정제완료.csv", index=False, encoding="utf-8-sig")
print(f"✅ 정제 완료! 새 이미지 표 기준 총 {len(final_df)}개 데이터가 4대 카테고리로 매핑되어 저장되었습니다.")

✅ 정제 완료! 새 이미지 표 기준 총 248019개 데이터가 4대 카테고리로 매핑되어 저장되었습니다.


### 4단계 정밀 매칭 파이프라인

In [28]:
%pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.6 MB/s  0:00:00m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [36]:
import re
import pandas as pd
from rapidfuzz import fuzz, process

# ---------------------------------------------------------
# Step 0. 정제된 식약처 파일 & CU 상품 파일 불러오기
# ---------------------------------------------------------
cu_df = pd.read_csv("cu_products.csv")
food_df = pd.read_csv("식약처_영양DB_정제_처음분류.csv")


# 1) CU 상품명 괄호 접두사 제거 함수 (도), 김), 햄), 빅삼) 등)
def clean_cu_name(name):
    if not isinstance(name, str):
        return ""
    # '도)', '빅삼)', '김)' 등 앞의 카테고리 기호 제거
    cleaned = re.sub(r"^[가-힣a-zA-Z0-9]{1,4}\)", "", name)
    return cleaned.strip()


cu_df["clean_name"] = cu_df["display_name"].apply(clean_cu_name)

# 2) BGF 전담 제조사 데이터만 미리 추출 (팀원 매칭 핵심)
#bgf_manufacturers = ["비지에프푸드", "BGF푸드", "비지에프푸드제주", "비지에프푸드전북"]

# 제조사명 컬럼에서 BGF 관련 제조사 행만 색인(Indexing)
# 개선: '비지에프' 또는 'BGF' 단어가 들어간 모든 제조사 자동 추출
bgf_food_df = food_df[
    food_df["제조사명"]
    .astype(str)
    .str.contains("비지에프|BGF", case=False, na=False)
].copy()

# ---------------------------------------------------------
# Step 1~4. 4단계 정밀 매칭 수행
# ---------------------------------------------------------
matched_results = []

for idx, cu_row in cu_df.iterrows():
    raw_name = cu_row["display_name"]
    clean_name = cu_row["clean_name"]
    price = cu_row["price"]
    event_type = cu_row.get("event_type", "NONE")

    match_type = None
    matched_food = None

    # [1단계] 완전 일치 (CU 정제명과 식약처 식품명이 100% 동일)
    exact_match = food_df[food_df["식품명"] == clean_name]
    if not exact_match.empty:
        match_type = "1. 완전 일치"
        matched_food = exact_match.iloc[0]

    # [2단계] 수식어/콜라보 키워드 정제 후 일치
    if match_type is None:
        sub_name = re.sub(
            r"(수향미|레벨업|더블|빅삼|트리플)", "", clean_name
        ).strip()
        sub_match = food_df[food_df["식품명"] == sub_name]
        if not sub_match.empty:
            match_type = "2. 브랜드/콜라보 정제 일치"
            matched_food = sub_match.iloc[0]

    # [3단계 & 4단계] BGF 전담 제조사 인증 매칭 (BGF푸드 전용 DB에서 탐색)
    if match_type is None and not bgf_food_df.empty:
        bgf_food_names = bgf_food_df["식품명"].dropna().tolist()
        best_match = process.extractOne(
            clean_name, bgf_food_names, scorer=fuzz.WRatio
        )

        # 유사도 55점 이상 시 BGF 제조사 인증 매칭으로 구제 확정
        if best_match and best_match[1] >= 55:
            match_type = "4. BGF 전담 제조사 인증 매칭"
            matched_food = bgf_food_df[
                bgf_food_df["식품명"] == best_match[0]
            ].iloc[0]

    # 매칭 결과 패킹
    if matched_food is not None:
        matched_results.append({
            "display_name": raw_name,
            "clean_name": clean_name,
            "match_type": match_type,
            "matched_food_name": matched_food["식품명"],
            "manufacturer": matched_food["제조사명"],
            "category": matched_food.get("category", "기타"),
            "sub_category": matched_food.get("식품소분류명", ""),
            "price": price,
            "event_type": event_type,
            "calories": matched_food.get("에너지(kcal)", 0),
            "protein": matched_food.get("단백질(g)", 0),
            "sodium": matched_food.get("나트륨(mg)", 0),
            "carbs": matched_food.get("탄수화물(g)", 0),
            "sugars": matched_food.get("당류(g)", 0),
            "fat": matched_food.get("지방(g)", 0),
        })

# 결과 DataFrame 변환
final_matched_df = pd.DataFrame(matched_results)

# 백엔드 연동용 최종 CSV 파일 저장
final_matched_df.to_csv(
    "cu_products_with_nutrition.csv", index=False, encoding="utf-8-sig"
)


# ---------------------------------------------------------
# 성과 리포트 출력
# ---------------------------------------------------------
print("======================================================")
print("🎯 [CU-식약처 정밀 매칭 성과 리포트]")
print("======================================================")
print(f"- 원본 CU 전체 상품 수: {len(cu_df)}건")
print(
    f"- 최종 매칭 성공 상품 수: {len(final_matched_df)}건"
    f" ({len(final_matched_df)/len(cu_df)*100:.1f}%)"
)
print("\n[매칭 단계별 기여 건수 분포]")
print(final_matched_df["match_type"].value_counts().sort_index())

print("\n\n[4단계 BGF 전담 제조사 인증 매칭 구제 샘플 확인]")
bgf_samples = final_matched_df[
    final_matched_df["match_type"] == "4. BGF 전담 제조사 인증 매칭"
].head(10)

for idx, row in bgf_samples.iterrows():
    print(
        f" [4. BGF 전담 제조사 인증 매칭] '{row['display_name']}'"
        f"\n └── 식약처: '{row['matched_food_name']}' | {row['manufacturer']}"
        f" | {row['calories']} kcal\n"
    )

🎯 [CU-식약처 정밀 매칭 성과 리포트]
- 원본 CU 전체 상품 수: 3546건
- 최종 매칭 성공 상품 수: 715건 (20.2%)

[매칭 단계별 기여 건수 분포]
match_type
1. 완전 일치               342
2. 브랜드/콜라보 정제 일치         3
4. BGF 전담 제조사 인증 매칭    370
Name: count, dtype: int64


[4단계 BGF 전담 제조사 인증 매칭 구제 샘플 확인]
 [4. BGF 전담 제조사 인증 매칭] '도)돈까스쏘야더블정식'
 └── 식약처: '돈까스&제육더블정식' | (주)비지에프푸드 전북 | 210 kcal

 [4. BGF 전담 제조사 인증 매칭] '김)한돈불고기마늘쫑김밥'
 └── 식약처: '마늘쫑한돈불고기김밥' | (주)비지에프푸드 전북 | 177 kcal

 [4. BGF 전담 제조사 인증 매칭] '주)너비아니전주비빔밥바'
 └── 식약처: '치즈너비아니' | (주)청탑/(주)비지에프푸드 전북 | 185 kcal

 [4. BGF 전담 제조사 인증 매칭] '햄)더블업숯불불고기버거'
 └── 식약처: '더블치즈불고기버거' | (주)비지에프푸드 | 263 kcal

 [4. BGF 전담 제조사 인증 매칭] '도)맵달크리미닭강정감자'
 └── 식약처: '크리미감자뇨끼' | (주)비지에프푸드전북 | 184 kcal

 [4. BGF 전담 제조사 인증 매칭] '도)치즈청양간장닭강정'
 └── 식약처: '닭강정유부' | (주)비지에프푸드 전북 | 198 kcal

 [4. BGF 전담 제조사 인증 매칭] '김)닭강정김밥콤보세트'
 └── 식약처: '닭강정유부' | (주)비지에프푸드 전북 | 198 kcal

 [4. BGF 전담 제조사 인증 매칭] '도)탕수육고추잡채트리플'
 └── 식약처: '팝컵탕수육' | (주)비지에프푸드제주 | 275 kcal

 [4. BGF 전담 제조사 인증 매칭] '햄)스파이시마요치킨버거'
 └── 식약처: '뉴스파이시치킨버거' | (주)비지에프푸드 |

### 식약처 데이터 정제 (범위 더 넓혀서 다시)
#### 처음 분류했던 대로 다시 분류

In [35]:
import pandas as pd

# 1. 식약처 원본 파일 불러오기
file_path = "20260929_가공식품DB_323197건 (1).xlsx"  # 원본 엑셀 파일명
df = pd.read_excel(file_path)

# 2. 지금 보내주신 '전체 소분류 표' 기준 매핑 딕셔너리 생성
category_map = {}

# [식사류]
meal_items = [
    "즉석조리식품",
    "즉석섭취식품",
    "건면",
    "숙면",
    "생면",
    "유탕면",
    "만두",
    "만두피",
    "떡류",
    "곡류가공품",
    "서류가공품",
    "간편조리세트",
    "신선편의식품",
    "카레(커리)",
    "카레(커리)분",
    "생식제품",
    "생식함유제품",
    "묵류",
    "전분가공품",
    "밀가루",
    "영양강화 밀가루",
    "전분",
    "기타 농산가공품",
    "열량 및 영양공급용 식품",
    "신장질환자용 식단형 식품",
    "당뇨환자용 식단형 식품",
    "암환자용 식단형 식품",
    "고혈압환자용 식단형 식품",
]
for item in meal_items:
    category_map[item] = "식사류"

# [사이드류]
side_items = [
    "양념육",
    "분쇄가공육제품",
    "갈비가공품",
    "햄",
    "프레스햄",
    "생햄",
    "발효소시지",
    "혼합소시지",
    "소시지",
    "어육소시지",
    "어육살",
    "연육",
    "조미건어포",
    "건어포",
    "기타 건포류",
    "양념젓갈",
    "젓갈",
    "액젓",
    "조미액젓",
    "피단",
    "전란액",
    "난백액",
    "난황액",
    "난백분",
    "난황분",
    "치즈",
    "가공치즈",
    "모조치즈",
    "두부",
    "가공두부",
    "두류가공품",
    "유바",
    "청국장",
    "땅콩버터",
    "기타 동물성가공식품",
    "기타 어육가공품",
    "어육반제품",
    "김치",
    "김칫속",
    "절임식품",
    "소스",
    "복합조미식품",
    "양조간장",
    "한식간장",
    "혼합간장",
    "산분해간장",
    "된장",
    "한식된장",
    "기타장류",
    "혼합장",
    "고추장",
    "춘장",
    "개량메주",
    "한식메주",
    "향신료조제품",
    "천연향신료",
    "고춧가루",
    "카레(커리) 관련 조미원료 외 소스류",
    "발효식초",
    "희석식초",
    "마요네즈",
    "토마토케첩",
    "참기름",
    "추출참깨유",
    "들기름",
    "추출들깨유",
    "올리브유",
    "콩기름(대두유)",
    "해바라기유",
    "채종유(유채유 또는 카놀라유)",
    "미강유(현미유)",
    "옥수수기름(옥배유)",
    "땅콩기름(낙화생유)",
    "목화씨기름(면실유)",
    "홍화유(사플라워유 또는 잇꽃유)",
    "고추씨기름",
    "야자유",
    "팜유류",
    "기타식물성유지",
    "혼합식용유",
    "기타 식용유지가공품",
    "가공유지",
    "향미유",
    "쇼트닝",
    "마가린",
    "설탕",
    "기타설탕",
    "기타과당",
    "과당",
    "올리고당",
    "올리고당가공품",
    "물엿",
    "기타엿",
    "가공소금",
    "기타소금",
    "정제소금",
    "태움·용융소금",
    "덱스트린",
    "식물성크림",
    "기타 가공 원료",
    "발효버터유",
    "버터",
    "가공버터",
    "버터오일",
    "어유",
    "식용돈지",
    "원료돈지",
    "식용우지",
    "원료우지",
]
for item in side_items:
    category_map[item] = "사이드류"

# [디저트류]
dessert_items = [
    "빵류",
    "과자",
    "캔디류",
    "초콜릿가공품",
    "초콜릿",
    "준초콜릿",
    "밀크초콜릿",
    "화이트초콜릿",
    "기타 코코아가공품",
    "아이스크림",
    "아이스밀크",
    "샤베트",
    "비유지방아이스크림",
    "저지방아이스크림",
    "아이스크림믹스",
    "아이스밀크믹스",
    "비유지방아이스크림믹스",
    "추잉껌",
    "땅콩 또는 견과류가공품",
    "코코아분말",
    "당절임",
    "잼",
    "기타잼",
    "벌꿀",
    "로열젤리제품",
    "로열젤리",
    "코코아매스",
    "코코아버터",
]
for item in dessert_items:
    category_map[item] = "디저트류"

# [음료류]
drink_items = [
    "혼합음료",
    "과·채주스",
    "과·채음료",
    "액상차",
    "침출차",
    "고형차",
    "음료베이스",
    "탄산음료",
    "탄산수",
    "기타발효음료",
    "유산균음료",
    "효모음료",
    "농축과·채즙(또는 과·채분)",
    "가공두유",
    "원액두유",
    "우유",
    "강화유유",
    "가공유",
    "유당분해유",
    "유산균첨가유",
    "발효유",
    "농후발효유",
    "농후크림발효유",
    "크림발효유",
    "탈지농축유",
    "환원유",
    "가공유크림",
    "유크림",
    "가공연유",
    "가당연유",
    "전지분유",
    "탈지분유",
    "혼합분유",
    "유청단백분말",
    "소주",
    "맥주",
    "탁주",
    "청주",
    "과실주",
    "일반증류주",
    "리큐르",
    "위스키",
    "수분 및 전해질보충용 조제식품",
    "일반 환자용 균형영양조제식품(균형영양조제식품)",
    "당뇨환자용 영양조제식품",
    "신장질환자용 영양조제식품",
    "암환자용 영양조제식품",
    "고혈압환자용 영양조제식품",
    "기타환자용 영양조제식품",
    "장질환자용 단백질수분해 영양조제식품",
    "선천성대사질환자용조제식품",
    "영아용 조제유",
    "성장기용 조제유",
    "영·유아용 특수조제식품",
]
for item in drink_items:
    category_map[item] = "음료류"


# 3. 정제 수행
subcat_col = "식품소분류명"
filtered_df = df[df[subcat_col].isin(category_map.keys())].copy()
filtered_df["category"] = filtered_df[subcat_col].map(category_map)

# 4. 핵심 컬럼 추출 (제조사명 포함)
target_columns = [
    "category",
    subcat_col,
    "식품명",
    "제조사명",  # 엑셀의 실제 제조사 컬럼명 확인
    "식품중량",
    "에너지(kcal)",
    "단백질(g)",
    "나트륨(mg)",
    "탄수화물(g)",
    "당류(g)",
    "지방(g)",
]

existing_cols = [c for c in target_columns if c in filtered_df.columns]
final_df = filtered_df[existing_cols]

# 5. 저장
final_df.to_csv("식약처_영양DB_정제_처음분류.csv", index=False, encoding="utf-8-sig")
print(
    f"✅ 전체 소분류 표 반영 완료! 총 {len(final_df)}개 데이터가 정제되어 저장되었습니다."
)

✅ 전체 소분류 표 반영 완료! 총 277070개 데이터가 정제되어 저장되었습니다.


### CU데이터 정제

In [37]:
import re
import pandas as pd
from rapidfuzz import fuzz, process

# ---------------------------------------------------------
# Step 1. CU 데이터 및 식약처 DB 전체 데이터 불러오기
# ---------------------------------------------------------
# 파일명은 보유하고 계신 원본 파일명으로 변경해 주세요.
cu_df = pd.read_csv("cu_products.csv")

# 식약처 원본 엑셀/CSV를 소분류 버림(isin) 없이 전체 불러옵니다.
food_df_raw = pd.read_excel(
    "20260929_가공식품DB_323197건 (1).xlsx"
)  # CSV일 경우 pd.read_csv("식약처_영양DB.csv")

# 핵심 컬럼만 정제하여 '식약처_영양DB_전체.csv' 생성
target_cols = [
    "식품소분류명",
    "식품명",
    "제조사명",
    "식품중량",
    "에너지(kcal)",
    "단백질(g)",
    "나트륨(mg)",
    "탄수화물(g)",
    "당류(g)",
    "지방(g)",
]
existing_cols = [c for c in target_cols if c in food_df_raw.columns]
food_df = food_df_raw[existing_cols].copy()

# 4대 넓은 카테고리(식사류, 사이드류, 디저트류, 음료류) 유연하게 자동 매핑
subcat_col = (
    "식품소분류명" if "식품소분류명" in food_df.columns else "소분류명"
)


def get_broad_category(subcat):
    if not isinstance(subcat, str):
        return "사이드류"
    if any(
        k in subcat
        for k in [
            "음료",
            "주스",
            "차",
            "우유",
            "유",
            "탄산",
            "두유",
            "워터",
            "커피",
            "주류",
            "소주",
            "맥주",
            "조제식품",
        ]
    ):
        return "음료류"
    elif any(
        k in subcat
        for k in [
            "빵",
            "과자",
            "캔디",
            "초콜릿",
            "아이스크림",
            "빙과",
            "껌",
            "잼",
            "젤리",
            "견과",
        ]
    ):
        return "디저트류"
    elif any(
        k in subcat
        for k in [
            "즉석",
            "면",
            "만두",
            "떡",
            "밥",
            "도시락",
            "김밥",
            "샌드위치",
            "햄버거",
            "카레",
            "시리얼",
        ]
    ):
        return "식사류"
    else:
        return "사이드류"


food_df["category"] = food_df[subcat_col].apply(get_broad_category)
food_df.to_csv("식약처_영양DB_전체.csv", index=False, encoding="utf-8-sig")


# ---------------------------------------------------------
# Step 2. CU 상품명 전처리 함수 (정제 규칙 강화)
# ---------------------------------------------------------
def clean_cu_name(name):
    if not isinstance(name, str):
        return ""
    # 1) '도)', '김)', '햄)', '빅삼)', '샌)' 등 카테고리 괄호 접두어 제거
    cleaned = re.sub(r"^[가-힣a-zA-Z0-9]{1,4}\)", "", name)
    # 2) [행사], (증정) 등 특수문자 괄호 제거
    cleaned = re.sub(r"\[.*?\]|\(.*?\)", "", cleaned)
    # 3) 특수문자 제거 및 양쪽 공백 정리
    cleaned = re.sub(r"[^\w\s가-힣a-zA-Z0-9]", "", cleaned)
    return cleaned.strip()


cu_df["clean_name"] = cu_df["display_name"].apply(clean_cu_name)

# BGF 전담 제조사 전용 DB 인덱싱 (3~4단계 구제용)
mfg_col = "제조사명" if "제조사명" in food_df.columns else "업소명"
bgf_food_df = food_df[
    food_df[mfg_col]
    .astype(str)
    .str.contains("비지에프|BGF", case=False, na=False)
].copy()


# ---------------------------------------------------------
# Step 3. 4단계 정밀 매칭 파이프라인 수행
# ---------------------------------------------------------
matched_results = []

for idx, cu_row in cu_df.iterrows():
    raw_name = cu_row["display_name"]
    clean_name = cu_row["clean_name"]
    price = cu_row["price"]
    event_type = cu_row.get("event_type", "NONE")

    match_type = None
    matched_food = None

    # [1단계] 완전 일치 (전체 식약처 DB 매칭)
    exact_match = food_df[food_df["식품명"] == clean_name]
    if not exact_match.empty:
        match_type = "1. 완전 일치"
        matched_food = exact_match.iloc[0]

    # [2단계] 브랜드/콜라보 키워드 정제 후 일치
    if match_type is None:
        sub_name = re.sub(
            r"(수향미|레벨업|더블|빅삼|트리플|연세)", "", clean_name
        ).strip()
        sub_match = food_df[food_df["식품명"] == sub_name]
        if not sub_match.empty:
            match_type = "2. 브랜드/콜라보 정제 일치"
            matched_food = sub_match.iloc[0]

    # [3단계 & 4단계] BGF 전담 제조사 인증 매칭 (신선식품 구제)
    if match_type is None and not bgf_food_df.empty:
        bgf_food_names = bgf_food_df["식품명"].dropna().tolist()
        best_match = process.extractOne(
            clean_name, bgf_food_names, scorer=fuzz.WRatio
        )

        if best_match and best_match[1] >= 55:
            match_type = "4. BGF 전담 제조사 인증 매칭"
            matched_food = bgf_food_df[
                bgf_food_df["식품명"] == best_match[0]
            ].iloc[0]

    # 매칭 성공 건 패킹
    if matched_food is not None:
        matched_results.append({
            "display_name": raw_name,
            "clean_name": clean_name,
            "match_type": match_type,
            "matched_food_name": matched_food["식품명"],
            "manufacturer": matched_food.get(mfg_col, "알수없음"),
            "category": matched_food.get("category", "사이드류"),
            "sub_category": matched_food.get(subcat_col, ""),
            "price": price,
            "event_type": event_type,
            "calories": matched_food.get("에너지(kcal)", 0),
            "protein": matched_food.get("단백질(g)", 0),
            "sodium": matched_food.get("나트륨(mg)", 0),
            "carbs": matched_food.get("탄수화물(g)", 0),
            "sugars": matched_food.get("당류(g)", 0),
            "fat": matched_food.get("지방(g)", 0),
        })

# 최종 결과 저장
final_matched_df = pd.DataFrame(matched_results)
final_matched_df.to_csv(
    "cu_products_with_nutrition.csv", index=False, encoding="utf-8-sig"
)


# ---------------------------------------------------------
# Step 4. 성과 리포트 출력
# ---------------------------------------------------------
print("======================================================")
print("🎯 [전체 DB 반영 CU-식약처 정밀 매칭 성과 리포트]")
print("======================================================")
print(f"- 원본 CU 전체 상품 수: {len(cu_df)}건")
print(
    f"- 최종 매칭 성공 상품 수: {len(final_matched_df)}건"
    f" ({len(final_matched_df)/len(cu_df)*100:.1f}%)"
)
print("\n[매칭 단계별 기여 건수 분포]")
print(final_matched_df["match_type"].value_counts().sort_index())

🎯 [전체 DB 반영 CU-식약처 정밀 매칭 성과 리포트]
- 원본 CU 전체 상품 수: 3546건
- 최종 매칭 성공 상품 수: 726건 (20.5%)

[매칭 단계별 기여 건수 분포]
match_type
1. 완전 일치               358
2. 브랜드/콜라보 정제 일치         5
4. BGF 전담 제조사 인증 매칭    363
Name: count, dtype: int64


In [1]:
import re
import pandas as pd
from rapidfuzz import fuzz, process

# 1. 파일 읽기 (파일명 확인 필수!)
cu_df = pd.read_csv("cu_products.csv")

# ⚠️ [핵심] 정제된 csv가 아니라, 식약처 원본 엑셀/CSV 전체를 직접 읽어옵니다.
# 엑셀 파일인 경우: pd.read_excel("식약처_영양DB.xlsx")
# CSV 파일인 경우: pd.read_csv("식약처_영양DB.csv") (파일명이 다르면 꼭 수정해 주세요!)
food_df = pd.read_excel("20260929_가공식품DB_323197건 (1).xlsx")

print(f"✅ 읽어온 식약처 전체 데이터 건수: {len(food_df)}건")
# 💡 여기서 위 출력 건수가 20만 건 이상으로 나와야 정상입니다!
# 만약 2만~3만 건으로 나온다면 파일명이 잘못되어 이전 정제 파일을 불러온 것입니다.


# 2. CU 상품명 정제 함수 (괄호 및 특수문자 제거)
def clean_cu_name(name):
    if not isinstance(name, str):
        return ""
    cleaned = re.sub(r"^[가-힣a-zA-Z0-9]{1,4}\)", "", name)
    cleaned = re.sub(r"\[.*?\]|\(.*?\)", "", cleaned)
    cleaned = re.sub(r"[^\w\s가-힣a-zA-Z0-9]", "", cleaned)
    return cleaned.strip()


cu_df["clean_name"] = cu_df["display_name"].apply(clean_cu_name)

# 3. 제조사 컬럼 찾기 및 BGF 전담 제조사 추출
mfg_col = "제조사명" if "제조사명" in food_df.columns else "업소명"
bgf_food_df = food_df[
    food_df[mfg_col]
    .astype(str)
    .str.contains("비지에프|BGF", case=False, na=False)
].copy()


# 4. 4단계 매칭 수행
matched_results = []

for idx, cu_row in cu_df.iterrows():
    raw_name = cu_row["display_name"]
    clean_name = cu_row["clean_name"]
    price = cu_row["price"]
    event_type = cu_row.get("event_type", "NONE")

    match_type = None
    matched_food = None

    # [1단계] 완전 일치 (24만건 전체 식약처 DB 대상)
    exact_match = food_df[food_df["식품명"] == clean_name]
    if not exact_match.empty:
        match_type = "1. 완전 일치"
        matched_food = exact_match.iloc[0]

    # [2단계] 수식어 정제 일치
    if match_type is None:
        sub_name = re.sub(
            r"(수향미|레벨업|더블|빅삼|트리플|연세)", "", clean_name
        ).strip()
        sub_match = food_df[food_df["식품명"] == sub_name]
        if not sub_match.empty:
            match_type = "2. 브랜드/콜라보 정제 일치"
            matched_food = sub_match.iloc[0]

    # [3~4단계] BGF 제조사 인증 매칭
    if match_type is None and not bgf_food_df.empty:
        bgf_food_names = bgf_food_df["식품명"].dropna().tolist()
        best_match = process.extractOne(
            clean_name, bgf_food_names, scorer=fuzz.WRatio
        )
        if best_match and best_match[1] >= 55:
            match_type = "4. BGF 전담 제조사 인증 매칭"
            matched_food = bgf_food_df[
                bgf_food_df["식품명"] == best_match[0]
            ].iloc[0]

    if matched_food is not None:
        matched_results.append({
            "display_name": raw_name,
            "clean_name": clean_name,
            "match_type": match_type,
            "matched_food_name": matched_food["식품명"],
            "manufacturer": matched_food.get(mfg_col, "알수없음"),
            "price": price,
            "event_type": event_type,
            "calories": matched_food.get("에너지(kcal)", 0),
            "protein": matched_food.get("단백질(g)", 0),
            "sodium": matched_food.get("나트륨(mg)", 0),
            "carbs": matched_food.get("탄수화물(g)", 0),
            "sugars": matched_food.get("당류(g)", 0),
            "fat": matched_food.get("지방(g)", 0),
        })

# 결과 출력
final_matched_df = pd.DataFrame(matched_results)
print("======================================================")
print("🎯 [전체 DB 반영 CU-식약처 정밀 매칭 성과 리포트]")
print("======================================================")
print(f"- 원본 CU 전체 상품 수: {len(cu_df)}건")
print(
    f"- 최종 매칭 성공 상품 수: {len(final_matched_df)}건"
    f" ({len(final_matched_df)/len(cu_df)*100:.1f}%)"
)
print("\n[매칭 단계별 기여 건수 분포]")
print(final_matched_df["match_type"].value_counts().sort_index())

✅ 읽어온 식약처 전체 데이터 건수: 323197건
🎯 [전체 DB 반영 CU-식약처 정밀 매칭 성과 리포트]
- 원본 CU 전체 상품 수: 3546건
- 최종 매칭 성공 상품 수: 726건 (20.5%)

[매칭 단계별 기여 건수 분포]
match_type
1. 완전 일치               358
2. 브랜드/콜라보 정제 일치         5
4. BGF 전담 제조사 인증 매칭    363
Name: count, dtype: int64


# 처음부터 다시!

#####

## 식약처 DB 1차 경량화

- Step 1. 식약처 DB 1차 경량화 (Filtering)
32만 건의 식약처 전체 데이터에는 농축수산물, 원재료, 외식 메뉴 등 편의점에서 팔지 않는 항목이 훨씬 많습니다. 이 데이터를 그대로 비교하면 연산 속도도 느려지고 오매칭 확률이 급증합니다.

실행 과제: 식약처 DB의 식품상세분류 또는 카테고리 컬럼을 확인하여 편의점 관련 항목(가공식품, 음료, 과자, 유제품, 빵류, 즉석섭취식품 등)만 필터링한 경량화 데이터셋을 별도로 저장합니다.

시행착오 포인트: "어디까지 남기고 어디까지 버릴 것인가?" (필터링 기준을 기록해두면 포트폴리오에서 좋은 백엔드 관점 스토리가 됩니다.)

- Step 2. CU 상품명 텍스트 노이즈 정제 (Preprocessing)
CU 데이터의 상품명에는 식약처 DB와 절대 일치할 수 없는 노이즈 정보가 섞여 있습니다.

제거 대상 노이즈:

행사 문구: [1+1], (증정), [행사], [기획]

브랜드/수식어: CU), PB, [단독]

규격/용량: 500ml, 100g, 2개입

실행 과제: Python의 정규표현식(re 모듈)을 이용해 전처리 함수 clean_text()를 작성합니다.

- Step 3. 규칙 기반(Rule-based) 1차 완전 매칭
텍스트 정제가 끝난 후, 연산량이 많은 유사도 알고리즘을 바로 돌리기 전에 확실한 1:1 관계부터 먼저 묶어주는 단계입니다.

실행 과제:

정제된 CU 상품명과 식약처 상품명이 100% 일치하는 건을 매칭하고, 결과 테이블에 match_type = '완전일치'로 기록합니다.

제조사 데이터가 있다면 제조사 + 정제 상품명 조건으로 2차 확정 매칭을 진행합니다.

목적: 약 10~20%의 데이터를 먼저 확정 지어 뒤이어 수행할 유사도 연산 대상을 줄입니다.

- Step 4. 유사도 기반(Fuzzy Match) 2차 매칭 & 임계값 설정
아까 화면에서 0건으로 나왔던 가장 핵심적인 단계입니다. 1차 매칭에서 남은 미매칭 데이터를 대상으로 문자열 유사도를 계산합니다.

실행 과제: rapidfuzz 라이브러리의 token_set_ratio 함수를 사용합니다.

이유: token_set_ratio는 단어 순서가 바뀌거나("CU 초코우유" vs "초코우유 CU") 중복 단어가 있어도 높은 점수를 줍니다.

임계값(Threshold) 실험:

점수 기준을 90점, 80점, 70점으로 각각 돌려보고 샘플 50개를 직접 눈으로 검수합니다.

예: 70점으로 낮췄더니 "딸기우유"와 "초코우유"가 매칭되는 오류(False Positive)가 발생하는지 확인하고, 최적의 임계값(예: 78점 또는 80점)을 최종 채택합니다.=

In [3]:
import pandas as pd

# 1. 파일 불러오기 (파일명은 실제 가지고 계신 파일명으로 변경해주세요)
# 엑셀 파일인 경우: pd.read_excel('파일명.xlsx')
# CSV 파일인 경우: pd.read_csv('파일명.csv')
df_food = pd.read_csv('20260929_가공식품DB_323197건 (1).csv')
df_cu = pd.read_csv('cu_products.csv')

# 2. 식약처 데이터 구조 확인
print("=== 식약처 데이터 컬럼 목록 ===")
print(df_food.columns.tolist())
print("\n=== 식약처 데이터 샘플 (2건) ===")
print(df_food.head(2))

print("\n" + "=" * 50 + "\n")

# 3. CU 데이터 구조 확인
print("=== CU 데이터 컬럼 목록 ===")
print(df_cu.columns.tolist())
print("\n=== CU 데이터 샘플 (2건) ===")
print(df_cu.head(2))

=== 식약처 데이터 컬럼 목록 ===
['식품코드', '식품명', '데이터구분코드', '데이터구분명', '식품기원코드', '식품기원명', '식품대분류코드', '식품대분류명', '대표식품코드', '대표식품명', '식품중분류코드', '식품중분류명', '식품소분류코드', '식품소분류명', '식품세분류코드', '식품세분류명', '영양성분함량기준량', '에너지(kcal)', '수분(g)', '단백질(g)', '지방(g)', '회분(g)', '탄수화물(g)', '당류(g)', '식이섬유(g)', '칼슘(mg)', '철(mg)', '인(mg)', '칼륨(mg)', '나트륨(mg)', '비타민 A(μg RAE)', '레티놀(μg)', '베타카로틴(μg)', '티아민(mg)', '리보플라빈(mg)', '니아신(mg)', '비타민 C(mg)', '비타민 D(μg)', '콜레스테롤(mg)', '포화지방산(g)', '트랜스지방산(g)', '니코틴산(mg)', '니코틴아마이드(mg)', '비오틴(μg)', '비타민 B6 / 피리독신(mg)', '비타민 B12(μg)', '엽산(μg DFE)', '콜린(mg)', '판토텐산(mg)', '비타민 D2(μg)', '비타민 D3(μg)', '비타민 E(mg α-TE)', '알파 토코페롤(mg)', '베타 토코페롤(mg)', '감마 토코페롤(mg)', '델타 토코페롤(mg)', '알파 토코트리에놀(mg)', '베타 토코트리에놀(mg)', '감마 토코트리에놀(mg)', '델타 토코트리에놀(mg)', '비타민 K(μg)', '비타민 K1(μg)', '비타민 K2(μg)', '갈락토오스(g)', '과당(g)', '당알콜(g)', '맥아당(g)', '알룰로오스(g)', '에리스리톨(g)', '유당(g)', '자당(g)', '타가토스(g)', '포도당(g)', '불포화지방산(g)', 'EPA와 DHA의 합(mg)', '가돌레산(20:1) / 에이코센산(20:1)(mg)', '감마 리놀렌산(18:3(n-6))(mg)', '네르본산(24:1)(mg)',

In [5]:
import re
import pandas as pd

# 1. 데이터 불러오기
df_food = pd.read_csv('20260929_가공식품DB_323197건 (1).csv', low_memory=False)
df_cu = pd.read_csv('cu_products.csv')

print(f"원본 식약처 데이터 전체 건수: {len(df_food)}건")

# 2. 가공식품 1차 안전 필터링
if '데이터구분명' in df_food.columns:
    df_food_sub = df_food[df_food['데이터구분명'] == '가공식품'].copy()
else:
    df_food_sub = df_food.copy()

# 3. 식약처 식품명 특수문자/노이즈 제거만 진행
df_food_sub['clean_food_name'] = df_food_sub['식품명'].astype(str).apply(
    lambda x: re.sub(r'[^\w\s]', '', x).strip()
)

print(f"안전하게 전처리된 식약처 가공식품 데이터: {len(df_food_sub)}건")

원본 식약처 데이터 전체 건수: 323197건
안전하게 전처리된 식약처 가공식품 데이터: 323197건


In [7]:
import re


def clean_cu_name(name):
    if not isinstance(name, str):
        return ""

    # 1. CU 특유의 카테고리 한 글자 접두어 제거 (예: '도)돈까스...' -> '돈까스...', '김)한돈...' -> '한돈...')
    text = re.sub(r"^[가-휘]\)", "", name)

    # 2. 괄호 및 괄호 안 내용 제거: [1+1], (증정), [단독] 등
    text = re.sub(r"\[.*?\]|\(.*?\)", "", text)

    # 3. 용량/규격 단위 제거 (예: 500ml, 100g, 2개입 등)
    text = re.sub(
        r"\d+\s*(ml|l|g|kg|kcal|개|입|개입)", "", text, flags=re.IGNORECASE
    )

    # 4. 특수문자 제거 및 공백 정제
    text = re.sub(r"[^\w\s]", " ", text)

    return text.strip()


# CU 데이터프레임에 정제 함수 적용
df_cu["clean_cu_name"] = df_cu["display_name"].apply(clean_cu_name)

# 전처리 전/후 10개 비교 출력
print("=== CU 상품명 전처리 결과 비교 ===")
print(df_cu[["display_name", "clean_cu_name"]].head(10))

=== CU 상품명 전처리 결과 비교 ===
   display_name clean_cu_name
0   도)돈까스쏘야더블정식     돈까스쏘야더블정식
1  김)한돈불고기마늘쫑김밥    한돈불고기마늘쫑김밥
2  주)너비아니전주비빔밥바    너비아니전주비빔밥바
3  햄)더블업숯불불고기버거    더블업숯불불고기버거
4  도)맵달크리미닭강정감자    맵달크리미닭강정감자
5   샌)미니한손쏙멘치에그     미니한손쏙멘치에그
6   도)치즈청양간장닭강정     치즈청양간장닭강정
7   김)닭강정김밥콤보세트     닭강정김밥콤보세트
8  도)탕수육고추잡채트리플    탕수육고추잡채트리플
9  햄)스파이시마요치킨버거    스파이시마요치킨버거


In [8]:
import time
from rapidfuzz import fuzz
import pandas as pd

# 1. 매칭 결과 저장용 리스트
matched_results = []

# 식약처 정제 명칭 및 원본 명칭 리스트 추출
food_clean_names = df_food_sub['clean_food_name'].tolist()

# 임계값 설정 (75점 이상이면 매칭 성공으로 간주)
SIMILARITY_THRESHOLD = 75

# 빠른 테스트를 위해 CU 상품 중 상위 100개만 먼저 매칭 테스트
# (전체 다 돌리려면 df_cu.head(100) 대신 df_cu 사용)
test_cu = df_cu.head(100)

start_time = time.time()

for idx, cu_row in test_cu.iterrows():
  cu_raw = cu_row['display_name']
  cu_clean = cu_row['clean_cu_name']

  # [1단계] 완전 일치 매칭 (Exact Match)
  exact_match = df_food_sub[df_food_sub['clean_food_name'] == cu_clean]

  if not exact_match.empty:
    matched_row = exact_match.iloc[0]
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': '1. 완전 일치',
        'score': 100,
        'energy_kcal': matched_row.get('에너지(kcal)', None),
    })
    continue

  # [2단계] 유사도 매칭 (Token Set Ratio)
  best_score = 0
  best_match_idx = -1

  for i, food_name in enumerate(food_clean_names):
    # 단어 순서 및 포함 관계에 유연한 token_set_ratio 사용
    score = fuzz.token_set_ratio(cu_clean, food_name)
    if score > best_score:
      best_score = score
      best_match_idx = i

    # 100점 나오면 더 이상 찾지 않고 조기 종료 (속도 최적화)
    if best_score == 100:
      break

  # 설정한 임계값(75점) 이상인 경우만 매칭
  if best_score >= SIMILARITY_THRESHOLD and best_match_idx != -1:
    matched_row = df_food_sub.iloc[best_match_idx]
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': f'2. 유사도 매칭 ({int(best_score)}점)',
        'score': best_score,
        'energy_kcal': matched_row.get('에너지(kcal)', None),
    })
  else:
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': None,
        'match_type': '3. 미매칭',
        'score': best_score,
        'energy_kcal': None,
    })

end_time = time.time()
df_result = pd.DataFrame(matched_results)

# 2. 결과 리포트 출력
total = len(df_result)
matched = len(df_result[df_result['match_type'] != '3. 미매칭'])

print('=' * 50)
print(f'소요 시간: {end_time - start_time:.2f}초')
print(
    f'매칭 성공률: {matched}/{total}건 ({matched / total * 100:.1f}%)'
)
print('=' * 50)
print('\n[매칭 유형별 건수]')
print(df_result['match_type'].value_counts())

print('\n=== 매칭 샘플 10건 확인 ===')
print(
    df_result[
        ['cu_raw', 'cu_clean', 'matched_food_name', 'match_type']
    ].head(10)
)

소요 시간: 16.67초
매칭 성공률: 54/100건 (54.0%)

[매칭 유형별 건수]
match_type
3. 미매칭              46
1. 완전 일치            14
2. 유사도 매칭 (88점)      8
2. 유사도 매칭 (75점)      5
2. 유사도 매칭 (77점)      4
2. 유사도 매칭 (80점)      4
2. 유사도 매칭 (93점)      4
2. 유사도 매칭 (76점)      3
2. 유사도 매칭 (85점)      2
2. 유사도 매칭 (82점)      2
2. 유사도 매칭 (100점)     2
2. 유사도 매칭 (90점)      2
2. 유사도 매칭 (87점)      2
2. 유사도 매칭 (95점)      1
2. 유사도 매칭 (86점)      1
Name: count, dtype: int64

=== 매칭 샘플 10건 확인 ===
         cu_raw    cu_clean matched_food_name       match_type
0   도)돈까스쏘야더블정식   돈까스쏘야더블정식        돈까스&제육더블정식  2. 유사도 매칭 (77점)
1  김)한돈불고기마늘쫑김밥  한돈불고기마늘쫑김밥          한돈불고기쌈김밥  2. 유사도 매칭 (77점)
2  주)너비아니전주비빔밥바  너비아니전주비빔밥바       너비아니전주비빔주먹밥  2. 유사도 매칭 (85점)
3  햄)더블업숯불불고기버거  더블업숯불불고기버거           더블불고기버거  2. 유사도 매칭 (82점)
4  도)맵달크리미닭강정감자  맵달크리미닭강정감자               NaN           3. 미매칭
5   샌)미니한손쏙멘치에그   미니한손쏙멘치에그               NaN           3. 미매칭
6   도)치즈청양간장닭강정   치즈청양간장닭강정               NaN           3. 미매칭
7   김)닭강정김밥콤보세트   닭강정김밥콤보세트        한끼닭강정&

In [9]:
import pandas as pd
from rapidfuzz import fuzz

# 1. BGF 전담 제조사 키워드 리스트 (팀원 코드의 4단계 로직 반영)
bgf_manufacturers = [
    'BGF',
    '비지에프',
    '델리로드',
    '푸드코아',
    '조이푸드',
    '시피케이',
    '영진เด일'
]

# 2. 매칭 결과 저장용 리스트
matched_results = []
SIMILARITY_THRESHOLD = 75  # 3단계 유사도 임계값

# 테스트용 (우선 상위 100개 상품 대상)
test_cu = df_cu.head(100)

for idx, cu_row in test_cu.iterrows():
  cu_raw = cu_row['display_name']
  cu_clean = cu_row['clean_cu_name']

  # [1단계] 완전 일치 매칭 (Exact Match)
  exact_match = df_food_sub[df_food_sub['clean_food_name'] == cu_clean]
  if not exact_match.empty:
    matched_row = exact_match.iloc[0]
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': '1. 완전 일치',
        'score': 100,
    })
    continue

  # [2단계] 브랜드/콜라보 전처리 일치 매칭
  # (브랜드명이나 특정 키워드를 떼어낸 후 다시 완전 일치 시도)
  # 예: '연세우유 생크림빵' -> '생크림빵' 변환 후 매칭
  clean_no_brand = cu_clean.replace('연세우유', '').strip()
  brand_match = df_food_sub[df_food_sub['clean_food_name'] == clean_no_brand]
  if clean_no_brand != cu_clean and not brand_match.empty:
    matched_row = brand_match.iloc[0]
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': '2. 브랜드/콜라보 정제 일치',
        'score': 95,
    })
    continue

  # [3단계] 유사도(Fuzzy) 매칭
  best_score = 0
  best_match_idx = -1

  for i, food_name in enumerate(df_food_sub['clean_food_name'].tolist()):
    score = fuzz.token_set_ratio(cu_clean, food_name)
    if score > best_score:
      best_score = score
      best_match_idx = i
    if best_score == 100:
      break

  if best_score >= SIMILARITY_THRESHOLD and best_match_idx != -1:
    matched_row = df_food_sub.iloc[best_match_idx]
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': f'3. 유사도 매칭 ({int(best_score)}점)',
        'score': best_score,
    })
    continue

  # [4단계] BGF 전담 제조사 인증 매칭
  # (제조사명 컬럼이 식약처 DB에 있는 경우 BGF 전담 제조사인지 교차 검증)
  bgf_matched = False
  if '제조사명' in df_food_sub.columns:
    for bgf_mfg in bgf_manufacturers:
      mfg_match = df_food_sub[
          (df_food_sub['제조사명'].astype(str).str.contains(bgf_mfg, na=False))
          & (df_food_sub['clean_food_name'].str.contains(cu_clean[:3]))
      ]
      if not mfg_match.empty:
        matched_row = mfg_match.iloc[0]
        matched_results.append({
            'cu_raw': cu_raw,
            'cu_clean': cu_clean,
            'matched_food_name': matched_row['식품명'],
            'match_type': '4. BGF 전담 제조사 인증 매칭',
            'score': 80,
        })
        bgf_matched = True
        break

  if bgf_matched:
    continue

  # 모든 단계 실패 시 미매칭
  matched_results.append({
      'cu_raw': cu_raw,
      'cu_clean': cu_clean,
      'matched_food_name': None,
      'match_type': '5. 미매칭',
      'score': best_score,
  })

# 결과 변환
df_result = pd.DataFrame(matched_results)
print(df_result['match_type'].value_counts())

match_type
5. 미매칭                 31
4. BGF 전담 제조사 인증 매칭    15
1. 완전 일치               14
3. 유사도 매칭 (88점)         8
3. 유사도 매칭 (75점)         5
3. 유사도 매칭 (77점)         4
3. 유사도 매칭 (80점)         4
3. 유사도 매칭 (93점)         4
3. 유사도 매칭 (76점)         3
3. 유사도 매칭 (85점)         2
3. 유사도 매칭 (82점)         2
3. 유사도 매칭 (100점)        2
3. 유사도 매칭 (90점)         2
3. 유사도 매칭 (87점)         2
3. 유사도 매칭 (95점)         1
3. 유사도 매칭 (86점)         1
Name: count, dtype: int64


#### 다시 한 결과
"단순 문자열 유사도(Text Similarity)의 한계를 극복하기 위해, 편의점 PB 전담 제조사(BGF) 메타데이터를 활용한 Cross-Validation 조건 분기를 파이프라인에 추가하여 미매칭률을 15%p 낮춤."

전체 CU 상품 데이터(3,500여 건) 전체를 대상으로 4단계 파이프라인(완전일치 -> 브랜드 정제 -> 유사도 매칭 -> BGF 제조사 인증)을 실행하는 풀 버전 코드

In [10]:
import time
import pandas as pd
from rapidfuzz import fuzz

# 1. BGF 전담 제조사 키워드 정의
bgf_manufacturers = [
    'BGF',
    '비지에프',
    '델리로드',
    '푸드코아',
    '조이푸드',
    '시피케이',
    '영진데일',
]

# 2. 매칭 결과 저장용 리스트
matched_results = []
SIMILARITY_THRESHOLD = 75  # 유사도 임계값

# 식약처 DB 정제 명칭 리스트 사전 추출 (연산 속도 최적화)
food_clean_names = df_food_sub['clean_food_name'].tolist()

print(
    f'🚀 전체 CU 상품 {len(df_cu)}건에 대해 4단계 매칭 파이프라인을 시작합니다...'
)
start_time = time.time()

for idx, cu_row in df_cu.iterrows():
  cu_raw = cu_row['display_name']
  cu_clean = cu_row['clean_cu_name']

  # [1단계] 완전 일치 매칭 (Exact Match)
  exact_match = df_food_sub[df_food_sub['clean_food_name'] == cu_clean]
  if not exact_match.empty:
    matched_row = exact_match.iloc[0]
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': '1. 완전 일치',
        'score': 100,
        'energy_kcal': matched_row.get('에너지(kcal)', None),
    })
    continue

  # [2단계] 브랜드/콜라보 정제 일치 매칭
  # 브랜드명 제거 후 재비교 (예시 키워드 제거)
  clean_no_brand = cu_clean.replace('연세우유', '').replace('뚱랑이', '').strip()
  if clean_no_brand != cu_clean:
    brand_match = df_food_sub[
        df_food_sub['clean_food_name'] == clean_no_brand
    ]
    if not brand_match.empty:
      matched_row = brand_match.iloc[0]
      matched_results.append({
          'cu_raw': cu_raw,
          'cu_clean': cu_clean,
          'matched_food_name': matched_row['식품명'],
          'match_type': '2. 브랜드/콜라보 정제 일치',
          'score': 95,
          'energy_kcal': matched_row.get('에너지(kcal)', None),
      })
      continue

  # [3단계] 유사도 매칭 (Token Set Ratio)
  best_score = 0
  best_match_idx = -1

  for i, food_name in enumerate(food_clean_names):
    score = fuzz.token_set_ratio(cu_clean, food_name)
    if score > best_score:
      best_score = score
      best_match_idx = i
    if best_score == 100:
      break

  if best_score >= SIMILARITY_THRESHOLD and best_match_idx != -1:
    matched_row = df_food_sub.iloc[best_match_idx]
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': '3. 유사도 매칭',
        'score': best_score,
        'energy_kcal': matched_row.get('에너지(kcal)', None),
    })
    continue

  # [4단계] BGF 전담 제조사 인증 매칭
  bgf_matched = False
  if '제조사명' in df_food_sub.columns and len(cu_clean) >= 3:
    for bgf_mfg in bgf_manufacturers:
      mfg_match = df_food_sub[
          (df_food_sub['제조사명'].astype(str).str.contains(bgf_mfg, na=False))
          & (df_food_sub['clean_food_name'].str.contains(cu_clean[:3]))
      ]
      if not mfg_match.empty:
        matched_row = mfg_match.iloc[0]
        matched_results.append({
            'cu_raw': cu_raw,
            'cu_clean': cu_clean,
            'matched_food_name': matched_row['식품명'],
            'match_type': '4. BGF 전담 제조사 인증 매칭',
            'score': 80,
            'energy_kcal': matched_row.get('에너지(kcal)', None),
        })
        bgf_matched = True
        break

  if bgf_matched:
    continue

  # 모든 조건 미충족 시 미매칭
  matched_results.append({
      'cu_raw': cu_raw,
      'cu_clean': cu_clean,
      'matched_food_name': None,
      'match_type': '5. 미매칭',
      'score': best_score,
      'energy_kcal': None,
  })

  # 진행상황 출력 (500건 단위)
  if (idx + 1) % 500 == 0:
    print(f"  - [{idx + 1}/{len(df_cu)}]건 처리 완료...")

end_time = time.time()
df_final_result = pd.DataFrame(matched_results)

# 3. 최종 결과 리포트 출력
total_cnt = len(df_final_result)
matched_cnt = len(
    df_final_result[df_final_result['match_type'] != '5. 미매칭']
)

print('=' * 60)
print(f'🎉 전체 매칭 완료! (총 소요 시간: {end_time - start_time:.2f}초)')
print(
    f'📊 전체 CU 상품 수: {total_cnt}건 | 최종 매칭 성공: {matched_cnt}건 ({matched_cnt / total_cnt * 100:.1f}%)'
)
print('=' * 60)
print('\n[단계별 기여 건수 및 비율]')
print(df_final_result['match_type'].value_counts())

🚀 전체 CU 상품 3546건에 대해 4단계 매칭 파이프라인을 시작합니다...
  - [500/3546]건 처리 완료...
  - [1000/3546]건 처리 완료...
  - [3000/3546]건 처리 완료...
  - [3500/3546]건 처리 완료...
🎉 전체 매칭 완료! (총 소요 시간: 1098.19초)
📊 전체 CU 상품 수: 3546건 | 최종 매칭 성공: 1462건 (41.2%)

[단계별 기여 건수 및 비율]
match_type
5. 미매칭                 2084
3. 유사도 매칭              1315
1. 완전 일치                107
4. BGF 전담 제조사 인증 매칭      40
Name: count, dtype: int64


In [11]:
import pandas as pd

# 데이터프레임이 df_final_result 로 저장되어 있는 상태에서 실행

print("=" * 80)
print("🔍 [1. 완전 일치 매칭 샘플 5건]")
print("=" * 80)
sample_exact = df_final_result[
    df_final_result['match_type'] == '1. 완전 일치'
].head(5)
print(
    sample_exact[['cu_raw', 'cu_clean', 'matched_food_name', 'energy_kcal']]
)

print("\n" + "=" * 80)
print("🔍 [3. 유사도 매칭 샘플 10건 (점수 및 매칭 결과 확인)]")
print("=" * 80)
sample_fuzzy = df_final_result[
    df_final_result['match_type'] == '3. 유사도 매칭'
].head(10)
print(
    sample_fuzzy[
        ['cu_raw', 'cu_clean', 'matched_food_name', 'score', 'energy_kcal']
    ]
)

print("\n" + "=" * 80)
print("🔍 [4. BGF 전담 제조사 인증 매칭 샘플 5건]")
print("=" * 80)
sample_bgf = df_final_result[
    df_final_result['match_type'] == '4. BGF 전담 제조사 인증 매칭'
].head(5)
print(sample_bgf[['cu_raw', 'cu_clean', 'matched_food_name', 'energy_kcal']])

print("\n" + "=" * 80)
print("🔍 [5. 미매칭 상품 샘플 10건 (왜 매칭 안됐는지 원인 분석용)]")
print("=" * 80)
sample_unmatched = df_final_result[
    df_final_result['match_type'] == '5. 미매칭'
].head(10)
print(sample_unmatched[['cu_raw', 'cu_clean', 'score']])

🔍 [1. 완전 일치 매칭 샘플 5건]
          cu_raw    cu_clean matched_food_name  energy_kcal
45   샐)저당라임유자샐러드   저당라임유자샐러드         저당라임유자샐러드         30.0
58  김)레벨업햄참치마요김밥  레벨업햄참치마요김밥        레벨업햄참치마요김밥        176.0
62      햄)불갈비롱버거      불갈비롱버거            불갈비롱버거        224.0
64    주)매콤양념족발밥바    매콤양념족발밥바          매콤양념족발밥바        187.0
65   도)백종원치킨3배한판   백종원치킨3배한판         백종원치킨3배한판        199.0

🔍 [3. 유사도 매칭 샘플 10건 (점수 및 매칭 결과 확인)]
          cu_raw    cu_clean matched_food_name       score  energy_kcal
0    도)돈까스쏘야더블정식   돈까스쏘야더블정식        돈까스&제육더블정식   77.777778        210.0
1   김)한돈불고기마늘쫑김밥  한돈불고기마늘쫑김밥          한돈불고기쌈김밥   77.777778        139.0
2   주)너비아니전주비빔밥바  너비아니전주비빔밥바       너비아니전주비빔주먹밥   85.714286        216.0
3   햄)더블업숯불불고기버거  더블업숯불불고기버거           더블불고기버거   82.352941        313.0
7    김)닭강정김밥콤보세트   닭강정김밥콤보세트        한끼닭강정&김밥세트   77.777778        213.0
9   햄)스파이시마요치킨버거  스파이시마요치킨버거          스파이시치킨버거   88.888889        269.0
12   김)수향미소불고기김밥   수향미소불고기김밥            소불고기김밥   80.000000        148.0
13  

In [12]:
import pandas as pd

# 1. 품목(카테고리)이 서로 다르게 매칭된 위험 사례 필터링
# 대표적인 주요 키워드 목록
keywords = [
    '샌드위치',
    '우유',
    '김밥',
    '도시락',
    '버거',
    '주스',
    '커피',
    '라면',
    '떡볶이',
    '빵',
]

mismatch_list = []

for idx, row in df_final_result.iterrows():
  cu_name = str(row['cu_clean'])
  food_name = str(row['matched_food_name'])

  if row['match_type'] == '5. 미매칭' or pd.isna(food_name):
    continue

  # CU 이름에는 키워드가 있는데, 매칭된 식약처 이름에는 해당 키워드가 없는 경우 (품목 오매칭)
  for kw in keywords:
    if kw in cu_name and kw not in food_name:
      mismatch_list.append(row)
      break

df_mismatch = pd.DataFrame(mismatch_list)

print(
    f'⚠️ 카테고리가 달라 오매칭으로 의심되는 건수: 총 {len(df_mismatch)}건'
)

# 오매칭 의심 샘플 10개 출력
print('\n[🚨 카테고리 불일치 오매칭 의심 사례 Top 10]')
print(
    df_mismatch[
        ['cu_clean', 'matched_food_name', 'match_type', 'score', 'energy_kcal']
    ].head(10)
)

# 2. 가장 위험한 '4. BGF 전담 제조사 인증 매칭' 결과 전체 확인
bgf_matches = df_final_result[
    df_final_result['match_type'] == '4. BGF 전담 제조사 인증 매칭'
]
print(f'\n[🚨 4단계 BGF 인증 매칭 샘플 (총 {len(bgf_matches)}건 중 10건)]')
print(
    bgf_matches[
        ['cu_clean', 'matched_food_name', 'score', 'energy_kcal']
    ].head(10)
)

⚠️ 카테고리가 달라 오매칭으로 의심되는 건수: 총 37건

[🚨 카테고리 불일치 오매칭 의심 사례 Top 10]
            cu_clean matched_food_name           match_type       score  \
38         샤샤샤우유크림샌드          샤샤샤요거트샌드  4. BGF 전담 제조사 인증 매칭   80.000000   
220        즉석빵 뺑오쇼콜라             뺑오쇼콜라            3. 유사도 매칭  100.000000   
289    즉석빵 촉촉한초코칩쿠키N          촉촉한초코칩쿠키            3. 유사도 매칭   76.190476   
342      스마일 커피빈커피캔디               스마일            3. 유사도 매칭  100.000000   
708      대두 한입딸기우유찰떡                대두            3. 유사도 매칭  100.000000   
732        대두 한입우유찰떡                대두            3. 유사도 매칭  100.000000   
981      스위트 트롤리미니버거             스위트 콘            3. 유사도 매칭   75.000000   
1244     노티드 아이스우유도넛       노티드 피크닉 마카롱  4. BGF 전담 제조사 인증 매칭   80.000000   
1499  PBICK 학교앞밀떡볶이컵          PBICK 우유            3. 유사도 매칭   76.923077   
1500   PBICK 국물밀떡볶이컵          PBICK 우유            3. 유사도 매칭   76.923077   

      energy_kcal  
38          190.0  
220         444.0  
289         460.0  
342         205.0  
708       

In [13]:
import time
import pandas as pd
from rapidfuzz import fuzz, process

# 1. 환경 설정 및 임계값
SIMILARITY_THRESHOLD = 75  # WRatio 기준 안전 임계값

# 주요 카테고리 키워드
categories = [
    '샌드위치',
    '우유',
    '김밥',
    '도시락',
    '버거',
    '주스',
    '커피',
    '라면',
    '떡볶이',
    '빵',
    '샐러드',
    '핫도그',
]


def extract_category(name):
  """상품명에서 카테고리 키워드를 추출하는 함수"""
  for cat in categories:
    if cat in str(name):
      return cat
  return None


# 2. 전처리: 카테고리 컬럼 생성 및 식약처 DB 사전(Dict) 생성
df_cu['category'] = df_cu['clean_cu_name'].apply(extract_category)
df_food_sub['category'] = df_food_sub['clean_food_name'].apply(
    extract_category
)

# [1단계 고속 매칭용] Dictionary 생성
food_exact_map = (
    df_food_sub.drop_duplicates('clean_food_name')
    .set_index('clean_food_name')
    .to_dict('index')
)

# 식약처 전체 명칭 및 카테고리별 명칭 리스트 준비
food_clean_names = df_food_sub['clean_food_name'].tolist()

matched_results = []
start_time = time.time()

print('🚀 개선된 안전 고속 매칭 파이프라인을 시작합니다...')

for idx, cu_row in df_cu.iterrows():
  cu_raw = cu_row['display_name']
  cu_clean = cu_row['clean_cu_name']
  cu_cat = cu_row['category']

  # [1단계] Dict 기반 완전 일치 (O(1) 속도)
  if cu_clean in food_exact_map:
    matched_row = food_exact_map[cu_clean]
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': '1. 완전 일치',
        'score': 100,
        'energy_kcal': matched_row.get('에너지(kcal)', None),
    })
    continue

  # [2단계] 카테고리 가드레일 기반 WRatio 유사도 매칭
  # CU 상품에 카테고리가 있다면, 해당 카테고리를 가진 식약처 DB 데이터만 후보로 필터링
  if cu_cat:
    candidate_df = df_food_sub[df_food_sub['category'] == cu_cat]
  else:
    candidate_df = df_food_sub

  candidate_names = candidate_df['clean_food_name'].tolist()

  if candidate_names:
    # process.extractOne + fuzz.WRatio 활용 (C++ 기반 고속 매칭)
    match_res = process.extractOne(
        cu_clean, candidate_names, scorer=fuzz.WRatio
    )

    if match_res and match_res[1] >= SIMILARITY_THRESHOLD:
      best_food_name, best_score, best_idx = match_res
      matched_row = candidate_df.iloc[best_idx]

      matched_results.append({
          'cu_raw': cu_raw,
          'cu_clean': cu_clean,
          'matched_food_name': matched_row['식품명'],
          'match_type': '2. 유사도 매칭 (WRatio)',
          'score': best_score,
          'energy_kcal': matched_row.get('에너지(kcal)', None),
      })
      continue

  # 조건 미충족 시 안전하게 미매칭 처리
  matched_results.append({
      'cu_raw': cu_raw,
      'cu_clean': cu_clean,
      'matched_food_name': None,
      'match_type': '3. 미매칭',
      'score': match_res[1] if (candidate_names and match_res) else 0,
      'energy_kcal': None,
  })

end_time = time.time()
df_new_result = pd.DataFrame(matched_results)

# 결과 리포트
total_cnt = len(df_new_result)
matched_cnt = len(df_new_result[df_new_result['match_type'] != '3. 미매칭'])

print('=' * 60)
print(f'⚡ 파이프라인 완료! (총 소요 시간: {end_time - start_time:.2f}초)')
print(
    f'📊 전체 CU 상품: {total_cnt}건 | 매칭 성공: {matched_cnt}건'
    f' ({matched_cnt / total_cnt * 100:.1f}%)'
)
print('=' * 60)
print(df_new_result['match_type'].value_counts())

🚀 개선된 안전 고속 매칭 파이프라인을 시작합니다...
⚡ 파이프라인 완료! (총 소요 시간: 8.31초)
📊 전체 CU 상품: 3546건 | 매칭 성공: 414건 (11.7%)
match_type
3. 미매칭                3132
2. 유사도 매칭 (WRatio)     307
1. 완전 일치               107
Name: count, dtype: int64


In [14]:
import re
import time
import pandas as pd
from rapidfuzz import fuzz, process

SIMILARITY_THRESHOLD = 65  # 현실적인 안전 임계값 조율


def clean_spec_words(text):
  """용량 단위(g, ml, L, kg) 및 숫자 정제 함수"""
  if not isinstance(text, str):
    return ''
  # 200g, 500ml, 1.5L 등 규격 표기 제거
  text = re.sub(
      r'\d+(\.\d+)?\s*(g|ml|l|kg|lgr|mg|kcal)',
      '',
      text,
      flags=re.IGNORECASE,
  )
  # 특수문자 제거 및 다중 공백 정리
  text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
  return ' '.join(text.split())


# 1. 전처리: 용량 및 규격 제거한 비교용 이름 컬럼 생성
df_cu['compare_name'] = df_cu['clean_cu_name'].apply(clean_spec_words)
df_food_sub['compare_name'] = df_food_sub['clean_food_name'].apply(
    clean_spec_words
)

# [1단계 고속 매칭용] Dictionary
food_exact_map = (
    df_food_sub.drop_duplicates('compare_name')
    .set_index('compare_name')
    .to_dict('index')
)
food_compare_names = df_food_sub['compare_name'].tolist()

matched_results = []
start_time = time.time()

print('🚀 매칭률 보완 고속 파이프라인을 시작합니다...')

for idx, cu_row in df_cu.iterrows():
  cu_raw = cu_row['display_name']
  cu_clean = cu_row['clean_cu_name']
  cu_compare = cu_row['compare_name']

  # [1단계] 규격 제거 후 완전 일치
  if cu_compare in food_exact_map:
    matched_row = food_exact_map[cu_compare]
    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': '1. 완전 일치 (정제 후)',
        'score': 100,
        'energy_kcal': matched_row.get('에너지(kcal)', None),
    })
    continue

  # [2단계] 전체 DB 대상 고속 유사도 매칭 (token_sort_ratio 적용)
  match_res = process.extractOne(
      cu_compare, food_compare_names, scorer=fuzz.token_sort_ratio
  )

  if match_res and match_res[1] >= SIMILARITY_THRESHOLD:
    best_food_name, best_score, best_idx = match_res
    matched_row = df_food_sub.iloc[best_idx]

    matched_results.append({
        'cu_raw': cu_raw,
        'cu_clean': cu_clean,
        'matched_food_name': matched_row['식품명'],
        'match_type': '2. 유연한 유사도 매칭',
        'score': best_score,
        'energy_kcal': matched_row.get('에너지(kcal)', None),
    })
    continue

  # 미매칭 처리
  matched_results.append({
      'cu_raw': cu_raw,
      'cu_clean': cu_clean,
      'matched_food_name': None,
      'match_type': '3. 미매칭',
      'score': match_res[1] if match_res else 0,
      'energy_kcal': None,
  })

end_time = time.time()
df_recovered_result = pd.DataFrame(matched_results)

# 결과 리포트
total_cnt = len(df_recovered_result)
matched_cnt = len(
    df_recovered_result[df_recovered_result['match_type'] != '3. 미매칭']
)

print('=' * 60)
print(f'⚡ 처리 완료! (소요 시간: {end_time - start_time:.2f}초)')
print(
    f'📊 전체 CU 상품: {total_cnt}건 | 매칭 성공: {matched_cnt}건'
    f' ({matched_cnt / total_cnt * 100:.1f}%)'
)
print('=' * 60)
print(df_recovered_result['match_type'].value_counts())

🚀 매칭률 보완 고속 파이프라인을 시작합니다...
⚡ 처리 완료! (소요 시간: 201.89초)
📊 전체 CU 상품: 3546건 | 매칭 성공: 2033건 (57.3%)
match_type
2. 유연한 유사도 매칭      1925
3. 미매칭             1513
1. 완전 일치 (정제 후)     108
Name: count, dtype: int64


In [15]:
import pandas as pd

# 1. 65점 이상 70점 이하 조건 만족 데이터 필터링
df_boundary = df_recovered_result[
    (df_recovered_result['score'] >= 65) & (df_recovered_result['score'] <= 70)
].copy()

print(
    f'🔍 65점~70점 사이의 경계선 매칭 건수: 총 {len(df_boundary)}건 / 전체 매칭'
    f' {matched_cnt}건 중'
)

# 2. 오매칭 여부 판단을 쉽게 돕기 위해 주요 정보 20개 샘플 추출
sample_20 = df_boundary[
    ['cu_raw', 'cu_clean', 'matched_food_name', 'score', 'energy_kcal']
].head(20)

# 출력 옵션 설정 (텍스트 잘림 방지)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

print('\n[🚨 65~70점 구간 매칭 데이터 샘플 20개]')
print(sample_20.to_string(index=False))

🔍 65점~70점 사이의 경계선 매칭 건수: 총 546건 / 전체 매칭 2033건 중

[🚨 65~70점 구간 매칭 데이터 샘플 20개]
      cu_raw     cu_clean matched_food_name     score  energy_kcal
 면)미트볼듬뿍토마토면    미트볼듬뿍토마토면         토마토소스 미트볼 66.666667        184.0
 도)제육볶음돈까스더블    제육볶음돈까스더블      김치제육&돈까스더블정식 70.000000        168.0
  김)레벨업햄참치제주     레벨업햄참치제주        레벨업햄참치마요김밥 66.666667        176.0
 샌)호밀햄치즈치킨반반    호밀햄치즈치킨반반         호밀햄치즈샌드위치 66.666667        226.0
 도)김치제육돈까스제주    김치제육돈까스제주      김치제육&돈까스더블정식 70.000000        168.0
 샌)서울저당고단백치킨    서울저당고단백치킨       저당고단백오리엔탈치킨 70.000000        200.0
 김)서울저당고단백치킨    서울저당고단백치킨       저당고단백오리엔탈치킨 70.000000        200.0
겟모닝)간장계란꼬마김밥 겟모닝 간장계란꼬마김밥      장조림간장계란밥꼬마김밥 66.666667        189.0
겟모닝)히팅햄치즈토스트 겟모닝 히팅햄치즈토스트         햄치즈토스트 스틱 66.666667        322.0
빅삼)쿵야김치볶음밥참치 빅삼 쿵야김치볶음밥참치         김치볶음밥참치마요 66.666667        204.0
 빅삼)전주비빔고추장빅  빅삼 전주비빔고추장빅         전주비빔고추장삼각 70.000000        149.0
  빅삼)간장우삼겹삼각   빅삼 간장우삼겹삼각        간장우삼겹BIG삼각 70.000000        189.0
   빅삼)바싹제육삼각    빅삼 바싹제육삼각         바싹제육BIG삼각 66.66666

In [16]:
import re
import time
import pandas as pd
from rapidfuzz import fuzz, process

# 1. 안전 임계값 상향 (신뢰성 확보)
SIMILARITY_THRESHOLD = 72

# 2. 전 품목 대상 카테고리 매핑 그룹 정의 (다대다 일치 검증용)
CATEGORY_GROUPS = {
    '면류': ['면', '스파게티', '파스타', '라면', '국수', '우동', '소바', '짬뽕', '짜장'],
    '밥류': ['밥', '김밥', '도시락', '초밥', '덮밥', '볶음밥', '삼각'],
    '빵/샌드위치': ['샌드위치', '버거', '빵', '토스트', '도넛', '모찌넛', '케익', '와플'],
    '육류/안주': ['미트볼', '닭강정', '치킨', '족발', '보쌈', '소세지', '바베큐'],
    '튀김/핫도그': ['핫도그', '튀김', '고로케', '붕어빵'],
    '음료/파우더': ['라떼', '파우더', '주스', '음료', '커피', '에이드', '우유', '차'],
}


def get_item_categories(text):
  """상품명에서 속한 모든 카테고리 그룹을 추출하는 함수"""
  matched_cats = set()
  for cat_name, keywords in CATEGORY_GROUPS.items():
    for kw in keywords:
      if kw in text:
        matched_cats.add(cat_name)
        break
  return matched_cats


def clean_cu_display_name(text):
  """CU 상품명의 접두어(면), 도) 등) 및 규격 단위 완벽 정제"""
  if not isinstance(text, str):
    return ''
  # 1. 괄호 접두어 제거 (예: 면)미트볼 -> 미트볼, 빅삼)전주비빔 -> 전주비빔)
  text = re.sub(r'^[가-힣0-9a-zA-Z]+\)', '', text)
  # 2. 용량 단위(g, ml, L, kg 등) 및 숫자로 된 규격 제거
  text = re.sub(
      r'\d+(\.\d+)?\s*(g|ml|l|kg|lgr|mg|kcal|cm)',
      '',
      text,
      flags=re.IGNORECASE,
  )
  # 3. 특수문자 제거 및 공백 정돈
  text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
  return ' '.join(text.split())


# --- 데이터 전처리 ---
df_cu['compare_name'] = df_cu['display_name'].apply(clean_cu_display_name)
df_food_sub['compare_name'] = df_food_sub['clean_food_name'].apply(
    clean_cu_display_name
)

# 고속 매칭용 구조 생성
food_compare_names = df_food_sub['compare_name'].tolist()

matched_results = []
start_time = time.time()

print('🚀 전 품목 신뢰성 보완 매칭 파이프라인 시작...')

for idx, cu_row in df_cu.iterrows():
  cu_raw = cu_row['display_name']
  cu_compare = cu_row['compare_name']
  cu_cats = get_item_categories(cu_compare)

  # 유사도 매칭 수행 (token_sort_ratio)
  match_res = process.extractOne(
      cu_compare, food_compare_names, scorer=fuzz.token_sort_ratio
  )

  if match_res:
    best_food_name, best_score, best_idx = match_res
    matched_row = df_food_sub.iloc[best_idx]
    matched_food_cats = get_item_categories(matched_row['compare_name'])

    # [가드레일 검증]
    # CU 상품에 카테고리가 정의되어 있다면, 식약처 DB 상품과 최소 1개 이상 카테고리가 교집합이어야 함
    category_check_passed = True
    if cu_cats:
      if not (cu_cats & matched_food_cats):  # 교집합이 없으면 차단
        category_check_passed = False

    # 점수 조건과 카테고리 가드레일을 모두 통과한 경우에만 매칭 성공 처리
    if best_score >= SIMILARITY_THRESHOLD and category_check_passed:
      matched_results.append({
          'cu_raw': cu_raw,
          'cu_clean': cu_compare,
          'matched_food_name': matched_row['식품명'],
          'match_type': '1. 안전 유사도 매칭',
          'score': best_score,
          'energy_kcal': matched_row.get('에너지(kcal)', None),
      })
      continue

  # 조건을 통과하지 못하면 오매칭 방지를 위해 안전하게 미매칭 처리
  matched_results.append({
      'cu_raw': cu_raw,
      'cu_clean': cu_compare,
      'matched_food_name': None,
      'match_type': '2. 미매칭 (안전 차단 포함)',
      'score': match_res[1] if match_res else 0,
      'energy_kcal': None,
  })

end_time = time.time()
df_safe_result = pd.DataFrame(matched_results)

# 결과 출력
total_cnt = len(df_safe_result)
matched_cnt = len(df_safe_result[df_safe_result['match_type'] != '2. 미매칭 (안전 차단 포함)'])

print('=' * 60)
print(f'⚡ 파이프라인 완료! (소요 시간: {end_time - start_time:.2f}초)')
print(
    f'📊 전체 CU 상품: {total_cnt}건 | 안전 매칭 성공: {matched_cnt}건'
    f' ({matched_cnt / total_cnt * 100:.1f}%)'
)
print('=' * 60)
print(df_safe_result['match_type'].value_counts())

🚀 전 품목 신뢰성 보완 매칭 파이프라인 시작...
⚡ 파이프라인 완료! (소요 시간: 182.95초)
📊 전체 CU 상품: 3546건 | 안전 매칭 성공: 2277건 (64.2%)
match_type
1. 안전 유사도 매칭         2277
2. 미매칭 (안전 차단 포함)    1269
Name: count, dtype: int64


In [2]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("nutrition_vlm_results.csv", encoding="utf-8-sig")

# 데이터 전체 확인
display(df)

,category_name,website_product_id,product_name,item_cd,list_image_url,on_item_no,pocketcu_product_name,nutrition_image_urls,image_files,status,error,img_file,item_id,열량,단백질,나트륨,탄수화물,당류,지방
0,음료,15504,빙그레)바나나우유240ml,8.800280e+07,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,빙그레)바나나우유240ml,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012700.png,2022030012700,208 kcal,7 g,110 mg,27 g,27 g,8 g
1,음료,903,롯데)에비앙P500ml,3.068320e+12,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,롯데)에비앙P500ml,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012774.png,2022030012774,54 kcal,NaN,4.4 mg,15.6 g,NaN,NaN
2,음료,3154,코카)몬스터에너지캔355ml,4.897037e+12,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,코카)몬스터에너지캔355ml,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012804.png,2022030012804,165 kcal,0 g,44 mg,41 g,41 g,0 g
3,음료,8908,코카)몬스터파이프캔355ml,4.897037e+12,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,코카)몬스터파이프캔355ml,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012806.png,2022030012806,56 kcal,0g,56mg,36g,35g,0g
4,음료,13357,코카)몬스터망고로코355ml,4.897037e+12,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,코카)몬스터망고로코355ml,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012807.png,2022030012807,175 kcal,1 g,61 mg,41 g,40 g,0.8 g
5,음료,7512,하이트)진로토닉워터300ml,8.801001e+12,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,하이트)진로토닉워터300ml,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012912.png,2022030012912,128kcal,0g,5mg,31g,31g,0.5g
6,음료,14543,하이트)토닉워터제로300ml,8.801001e+12,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,하이트)토닉워터제로300ml,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012913.png,2022030012913,10kcal,0g,70mg,9.7g,2g,0g
7,음료,6043,하이트)블랙보리P520ml,8.801001e+12,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,하이트)블랙보리P520ml,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012914.png,2022030012914,34 kcal,1 g,15 mg,8 g,2 g,0 g
8,음료,8595,하이트)블랙보리라이트520,8.801001e+12,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,하이트)블랙보리라이트520,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012915.png,2022030012915,0 kcal,0 g,35 mg,1 g 미만,0 g,0 g
9,음료,814,HK)컨디션헛개병,8.801007e+12,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.022030e+12,HK)컨디션헛개병100ml,https://arqachylpmku8348141.edge.naverncp.com/...,D:\Work\llm_project_new\cu_output\images\06_음료...,OK,NaN,2022030012938.png,2022030012938,60 kcal,2 g,45 mg,13 g,11 g,0 g


In [3]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("nutrition_vlm_results_01.csv", encoding="utf-8-sig")

# 데이터 전체 확인
display(df)

,category_name,website_product_id,product_name,price,item_cd,list_image_url,on_item_no,pocketcu_product_name,nutrition_image_urls,image_files,status,error,img_file,item_id,열량,단백질,나트륨,탄수화물,당류,지방
0,간편식사,18540,샐)뉴에그가먼저닭샐러드,3800,8801068916737,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.023070e+12,샐)뉴에그가먼저닭샐러드,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2023070031121.png,2023070031121,197 kcal,9 g,360 mg,11 g,3 g,13 g
1,간편식사,18608,샐)뉴치킨찹찹샐러드,3800,8801068916720,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.023080e+12,샐)뉴치킨찹찹샐러드,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2023080031266.png,2023080031266,191 kcal,13 g,350 mg,10 g,5 g,11 g
2,간편식사,17621,샐)허니리코타치즈샐러드,4800,8809148599016,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.023080e+12,샐)허니리코타치즈샐러드,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2023080032503.png,2023080032503,205 kcal,4.41g,187.28mg,23.54g,10.74g,10.24g
3,간편식사,17620,샐)오리지널닭가슴살샐러,4800,8809148599009,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.023080e+12,샐)오리지널닭가슴살샐러,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2023080032504.png,2023080032504,105 kcal,8.03g,676.55mg,15.03g,6.68g,1.64g
4,간편식사,23571,샐)블랙페퍼닭가슴살,4900,8809954751011,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.025040e+12,샐)블랙페퍼닭가슴살,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2025040048977.png,2025040048977,170 kcal,17.39g,622.27mg,15.59g,7%,6.94g
5,간편식사,25777,삼)치킨마요삼각,1400,8801771035527,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.026010e+12,주)뉴리치치킨마요삼각,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2026010060390.png,2026010060390,190kcal,5g,390mg,32g,0g,4.7g
6,간편식사,24907,도)급식대가뉴정석도시락,5700,8801771034643,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.026010e+12,도)급식대가뉴정석도시락,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2026010060411.png,2026010060411,735kcal,38g,"1,440mg",112g,17g,15g
7,간편식사,26413,삼)참치마요닭갈비더블,2300,8801771034476,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.026010e+12,주)참치마요닭갈비더블,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2026010060415.png,2026010060415,347 kcal,9g,800mg,62g,2%,7g
8,간편식사,21724,김)불고기김밥,3400,8801771031567,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.026010e+12,김)압도적불고기김밥,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2026010060483.png,2026010060483,454Kcal,19g,850mg,63g,7g,14g
9,간편식사,25854,도)오늘의9첩한상,5900,8800336392778,https://tqklhszfkvzk6518638.edge.naverncp.com/...,2.026010e+12,도)정성가득9첩한상,https://arqachylpmku8348141.edge.naverncp.com/...,/home/ai/llm-project/cu_data/cu_output/images/...,OK,NaN,2026010060773.png,2026010060773,714kcal,26g,"1,130mg",94g,10g,26g


## 1단계: VLM 결과 파싱 및 데이터 정제 파이프라인 구축

- 1단계: 데이터 정제 파이프라인 (01_parse_data.py)
- 2단계: RAG / 추천 AI 에이전트 서비스 로직(02_recommend_agent.py)